### # Data Profiling – Transaction Items

This notebook profiles the transaction_items dataset to understand
row-level granularity, relationships with transactions, and data quality
across all monthly files.


In [0]:
# reading all files into a dataframe
df = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/coffee_raw_volume/transaction_items/")
)

df.printSchema()
df.show(5)


In [0]:
# Row count & uniqueness
from pyspark.sql.functions import count, countDistinct

df.select(
    count("*").alias("total_rows"),
    countDistinct("transaction_id").alias("distinct_transaction_ids"),
    countDistinct("item_id").alias("distinct_item_ids")
).show()



In [0]:
# items per transaction
df.groupBy("transaction_id") \
  .count() \
  .describe() \
  .show()


In [0]:
# Time range validation 
from pyspark.sql.functions import min, max

df.select(
    min("created_at").alias("min_created_at"),
    max("created_at").alias("max_created_at")
).show()


In [0]:
# Null checks
df.selectExpr(
    "sum(case when transaction_id is null then 1 else 0 end) as null_transaction_id",
    "sum(case when item_id is null then 1 else 0 end) as null_item_id",
    "sum(case when quantity is null then 1 else 0 end) as null_quantity",
    "sum(case when unit_price is null then 1 else 0 end) as null_unit_price"
).show()


In [0]:
# Quantity and price checks
df.select(
    "quantity",
    "unit_price",
    "subtotal"
).describe().show()


###  Key Observations

- transaction_items represents line-level data with multiple records per transaction.
- total rows exceed the number of distinct transaction_ids, confirming one-to-many relationship.
- item_id values align with the menu_items dataset.
- Quantity and price values fall within expected ranges.
- Subtotal values are consistent with quantity and unit price.
- created_at aligns with the transactions dataset time range.
- No critical null values observed in key columns.
